# Capstone: does this pair want a second date?

MichAl Academy, lesson 2.26.

Between 2002 and 2004, researchers at Columbia ran speed dating nights. Everyone met
everyone for four minutes, then privately ticked yes or no. A **match** is when
both people ticked yes.

You get 8,378 pairs, with what each person said about the other: how attractive,
how funny, how sincere, what they claimed to be looking for, how much they
thought the other person liked them.

**Your job is not to get the highest score.** It is to end up able to answer one
question honestly: *if the organiser ran another night, what would you actually
tell them to do?*

Work the tasks in order. Each has a check cell that tells you when you have it.
Answers are folded at the bottom, and reading them early wastes the exercise.

Budget about ninety minutes.


In [ ]:
import warnings

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (average_precision_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

data = fetch_openml("SpeedDating", version=1, as_frame=True)
pairs, matched = data.data, data.target.astype(int)

print(f"{len(pairs)} pairs, {pairs.shape[1]} columns")
print(pairs[["gender", "age", "race", "attractive_partner",
             "funny_partner", "shared_interests_partner"]].head())


## Task 1: what does doing nothing score?

Before any model. Two numbers: how often a pair actually matched, and what
accuracy you get by predicting "no match" every single time.

Lesson 2.11.3 is why this has to come first.


In [ ]:
base_rate     = None      # TODO: the fraction of pairs that matched
lazy_accuracy = None      # TODO: accuracy of always answering "no match"

print("base rate     :", base_rate)
print("always say no :", lazy_accuracy)
print()
print("task 1 done?", lazy_accuracy is not None and lazy_accuracy > 0.8)


## Task 2: is any column too good to be true?

Some columns were recorded at the same event as the answer. Whether that makes
them a leak depends entirely on **when you would be making the prediction**, and
that is a decision you have to make rather than look up.

Score a few candidates on their own and see which ones carry real signal.

Lesson 2.3.1 is the one about leakage.


In [ ]:
# `like` is how much this person said they liked the other. `guess_prob_liked`
# is whether they thought it was mutual.
for col in ["like", "guess_prob_liked", "attractive_partner", "age"]:
    ok = pairs[col].notna()
    auc = roc_auc_score(matched[ok], pairs.loc[ok, col])
    print(f"{col:22s} alone, ROC-AUC {auc:.4f}")


In [ ]:
would_you_have_it = None   # TODO: True or False. At prediction time, do you
                           #       know how much each person liked the other?
why               = ""     # TODO: one sentence saying when it is a leak and
                           #       when it is not

print("keep it?", would_you_have_it)
print("why     :", why)
print()
print("task 2 done?", would_you_have_it is not None and len(why) > 20)


## Task 3: fit one model and measure it two ways

One model, fitted once, scored out of fold so no pair is judged by a model that
trained on it. This cell does the fitting because it takes a minute; the
measuring is yours.


In [ ]:
categorical = pairs.select_dtypes(exclude="number").columns
numeric = pairs.select_dtypes(include="number").columns

model = make_pipeline(
    ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value",
                               unknown_value=-1), list(categorical)),
        ("num", "passthrough", list(numeric))]),
    HistGradientBoostingClassifier(random_state=0))

score = cross_val_predict(model, pairs, matched,
                          cv=StratifiedKFold(5, shuffle=True, random_state=0),
                          method="predict_proba")[:, 1]

print("one score per pair, between 0 and 1:", score[:5].round(3))


In [ ]:
accuracy   = None      # TODO: accuracy of (score > 0.5) against matched
lift       = None      # TODO: how much that beats task 1's lazy_accuracy
avg_prec   = None      # TODO: average_precision_score(matched, score)
ap_random  = None      # TODO: what average precision a random model would get

print(f"accuracy          : {accuracy}")
print(f"lift over doing nothing: {lift}")
print(f"average precision : {avg_prec}  against {ap_random} for random")
print()
print("task 3 done?", lift is not None and avg_prec is not None)


Stop and look at those two numbers before moving on. One of them says the model
is barely worth having and the other says it is clearly real. Both are correct.
Lesson 2.11.2 and lesson 2.14.2 are the two sides of it.

## Task 4: the organiser can arrange 1,000 second dates

Not 8,378. One thousand.

Find the threshold that produces about a thousand predicted matches, and say
what that buys: how many real matches are in those thousand, and what share of
all the real matches you found.

Lesson 2.16.1.


In [ ]:
# TODO: fill in precision and recall at each threshold
print(" threshold  predicted  precision  recall")
for th in (0.50, 0.40, 0.30, 0.25, 0.20, 0.15):
    predicted = (score > th).astype(int)
    prec = None      # TODO
    rec = None       # TODO
    print(f" {th:9.2f}  {predicted.sum():9d}  {prec}      {rec}")


In [ ]:
chosen_threshold = None    # TODO: the one closest to a 1,000 date budget
real_matches_found = None  # TODO: how many of those predictions were real

print("threshold        :", chosen_threshold)
print("real matches     :", real_matches_found)
print()
print("task 4 done?", chosen_threshold is not None and real_matches_found is not None)


## Task 5: who does it work worse for?

Same model, same threshold, split the pairs by `race` and measure recall in each
group. Then look at the base rate in each group as well, because lesson 2.18.3
says those two cannot both be equalised.


In [ ]:
# Do task 4 first. Until chosen_threshold is set this falls back to the library
# default of 0.50, which is not your answer, and the table below will change
# when you set it.
threshold = chosen_threshold if chosen_threshold is not None else 0.50
predicted = (score > threshold).astype(int)

print(f"at a threshold of {threshold}")
print(f"{'group':28s} {'n':>6s} {'base':>7s} {'recall':>8s} {'precision':>10s}")
for group, idx in pairs.groupby("race", observed=True).groups.items():
    idx = np.array(list(idx))
    if len(idx) < 200:
        continue
    print(f"{str(group)[:26]:28s} {len(idx):6d} "
          f"{matched.iloc[idx].mean():7.4f} "
          f"{recall_score(matched.iloc[idx], predicted[idx]):8.4f} "
          f"{precision_score(matched.iloc[idx], predicted[idx], zero_division=0):10.4f}")


In [ ]:
worst_group_recall = None   # TODO: the lowest group recall in that table
best_group_recall  = None   # TODO: the highest
gap                = None   # TODO: the difference

print("gap in recall between groups:", gap)
print()
print("task 5 done?", gap is not None and gap > 0.05)


Whatever you found there, nothing in it says the model is unacceptable. It says
a decision exists, it is not a technical one, and it gets made by default if
nobody makes it deliberately.

## Task 6: what is the model actually using?

Ask which columns it leans on, then decide whether you would be comfortable
saying that out loud to the people who filled in the forms.

Lesson 2.17.1.


In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(pairs, matched, test_size=0.3,
                                      stratify=matched, random_state=0)
fitted = model.fit(Xtr, ytr)
imp = permutation_importance(fitted, Xte, yte, n_repeats=3, random_state=0,
                             scoring="average_precision", n_jobs=1)

top = np.argsort(imp.importances_mean)[::-1][:8]
for i in top:
    print(f"{pairs.columns[i]:30s} {imp.importances_mean[i]:+.4f}")


In [ ]:
top_feature = None      # TODO: the column at the top of that list
comfortable = None      # TODO: True or False, and be able to defend it
note        = ""        # TODO: one sentence on what that column measures and
                        #       what it does not

print("top feature :", top_feature)
print("comfortable :", comfortable)
print("note        :", note)
print()
print("task 6 done?", top_feature is not None and len(note) > 20)


## Write it up

The deliverable is not a model. It is a one-page note to the organiser that a
non-technical person can act on, containing:

- what you would do differently at the next event, in plain sentences
- how many second dates to arrange, and what fraction of them will work out
- which group the system serves worst, and what you propose about it
- one thing you do not trust about any of this

"Average precision 0.6054" is not a sentence for that note. "Arrange a thousand
second dates from the top of the list and about six hundred of them go well,
against a hundred and sixty if you picked at random" is.

## If you want to push it further

1. Refit without `like` and `guess_prob_liked` and see what the model is worth
   when it only knows what was on the form before the event. That is the honest
   version of the "before the date" use case from task 2.
2. Change the budget from 1,000 to 600 and to 2,000. The threshold moves and
   every number in task 5's table moves with it. Predict, before you run it,
   whether the gap between groups shrinks, grows or stays put.
3. Split by `samerace` instead of `race` and see whether the model has learned
   something about the event or something about people.


## Answers

Only after you have your own.

<details>
<summary>Task 1: the baseline</summary>

```python
base_rate = matched.mean()
lazy_accuracy = 1 - base_rate
```

**0.1647 and 0.8353.** One pair in six matched, so predicting "no match" for
every pair on the night is right 83.5% of the time and arranges zero dates. Any
accuracy you report later has to be read against that number, which is lesson
2.11.3.

</details>

<details>
<summary>Task 2: too good to be true</summary>

`like` alone gives **ROC-AUC 0.7432**, and `guess_prob_liked` alone gives
**0.6927**. For comparison, `attractive_partner` gives less and `age` gives
almost nothing.

There is no single right answer, which is the point.

**If you are predicting after the event**, both are legitimate: they are answers
on a form you have. **If you are predicting before the event**, from a profile,
neither exists yet, and a model trained with them will look excellent in
development and fail completely in use. That is lesson 2.3.1's definition of
leakage exactly: information that will not be there when the prediction is
really made.

The habit to take away is that "is this a leak" is never answered by looking at
the column. It is answered by saying out loud when the prediction happens.

</details>

<details>
<summary>Task 3: the two measurements</summary>

```python
accuracy  = ((score > 0.5).astype(int) == matched).mean()
lift      = accuracy - lazy_accuracy
avg_prec  = average_precision_score(matched, score)
ap_random = matched.mean()
```

**Accuracy 0.8687 against a baseline of 0.8353, so the lift is +0.0334.** Read
alone, that says the model is nearly worthless.

**Average precision 0.6054 against 0.1647 for random**, which is 3.7 times
better than chance. Read alone, that says the model is clearly real.

Both numbers are correct and they are measuring different things. Accuracy is
dominated by the 83.5% of pairs that did not match and that the model gets right
by agreeing with the baseline. Average precision asks only about the ranking of
the pairs that did. Lesson 2.11.2 and lesson 2.14.2.

</details>

<details>
<summary>Task 4: the threshold</summary>

```python
prec = precision_score(matched, predicted)
rec  = recall_score(matched, predicted)
```

| threshold | predicted | precision | recall |
|---|---|---|---|
| 0.50 | 798 | 0.6754 | 0.3906 |
| 0.40 | 1,106 | 0.6103 | 0.4891 |
| 0.30 | 1,474 | 0.5543 | 0.5920 |
| 0.25 | 1,709 | 0.5237 | 0.6486 |
| 0.20 | 2,000 | 0.4885 | 0.7080 |
| 0.15 | 2,410 | 0.4440 | 0.7754 |

**0.40 is the one that fits.** 1,106 second dates, of which **675 are real
matches**, and that is 48.9% of every real match in the data.

The sentence for the organiser: arrange about eleven hundred second dates and
six in ten go well. Picking eleven hundred pairs at random would give you about
180.

Notice that the library's default of 0.5 arranges only 798 dates and leaves 300
capacity unused. Nothing chose 0.5 for this problem; it is what
`predict` does when nobody tells it otherwise.

</details>

<details>
<summary>Task 5: the groups</summary>

At the threshold of 0.40 that task 4 arrives at:

| group | n | base rate | recall | precision |
|---|---|---|---|---|
| Asian/Pacific Islander | 1,982 | 0.1347 | **0.3708** | 0.5351 |
| Black/African American | 420 | 0.2024 | 0.4353 | 0.5968 |
| Latino/Hispanic American | 664 | 0.1852 | 0.4715 | 0.6105 |
| European/Caucasian American | 4,727 | 0.1667 | **0.5330** | 0.6278 |
| Other | 522 | 0.1973 | 0.5146 | 0.6463 |

Recall runs from **0.3708 to 0.5330**, a gap of **16 points** between the group
the model serves worst and the one it serves best. Precision is closer, 0.535 to
0.646.

The base rates differ too, 0.1347 against 0.1667, and that is the only condition
lesson 2.18.3's impossibility result needs. You cannot equalise recall and
precision across groups at once when the base rates differ. You can pick which
one to equalise, and you can decline to pick, in which case the threshold picks
for you.

**The gap is not something a different budget fixes.** Measured at four
thresholds, the recall gap is 0.142 at 587 dates, 0.162 at 1,106, 0.163 at 1,474
and 0.136 at 2,000. It moves by about two points across a range where the number
of dates more than triples.

Nothing here makes the model unusable. It makes a decision visible.

</details>

<details>
<summary>Task 6: what it uses</summary>

The top two by permutation importance are **`attractive_o`** and
**`attractive_partner`**, the two attractiveness ratings, and they come first
and second ahead of every stated preference on the form.

What that means, carefully: on this data, how attractive each person rated the
other predicts a match better than anything either of them said they were
looking for. That is a finding about four-minute conversations at a speed dating
event in 2002, measured on 8,378 pairs. It is not a finding about people in
general, and a model cannot tell you the difference.

The thing worth carrying out of this: the model learned what predicted the
answer in the data it was given. Whether that is the thing you wanted it to
learn is not a question the model can answer, and it is the question lesson
2.17.1 exists for.

</details>
